# NaverCafe Weekly Brand Summary (English Version)
Generates weekly GPT-4 summaries of community discussions per brand.

> **Note:** Original pipeline processes Korean community data.
> This version uses English-transliterated keywords for portfolio readability.

**Run schedule:** Every Tuesday

**Pipeline steps:**
1. Verify today is Tuesday
2. Fetch posts from previous Sunday to Saturday
3. Generate weekly summary per brand using GPT-4
4. Upload summaries to data warehouse

In [ ]:
# pip install snowflake-connector-python snowflake-sqlalchemy==1.5.1 openai==0.28

In [ ]:
import os
import re
import time
import pandas as pd
import datetime
import openai
import snowflake.connector
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# ── BRAND KEYWORD DICTIONARY ──────────────────────────────────────────────
# Maps each brand to known product line names and aliases (English)
# Note: Original pipeline ran on Korean community data;
# this version uses English-transliterated equivalents for readability.

BRAND_KEYWORDS = {
    'Huggies':  '|'.join(['Huggies', 'MaxDry', 'MagicDry', 'NatureMade',
                           'MagicComfort', 'NatureMadeOrganic', 'NatureSummer']),
    'Pampers':  '|'.join(['Pampers', 'Armoni', 'BabyDry', 'TouchOfNature',
                           'AirChaCha', 'NightPants']),
    'Penelope': '|'.join(['Penelope', 'MiracleAllDay', 'ThinLight']),
    'Mamipoko': '|'.join(['Mamipoko', 'AirFit', 'GoldBreathable', 'LeafGanic']),
    'Bosomi':   '|'.join(['Bosomi', 'WonderByWonder', 'MegaDry', 'RealCottonOrganic']),
    'Kindoh':   '|'.join(['Kindoh', 'UpAndPlay', 'AllDay', 'SlimFit'])
}

TARGET_BRANDS = list(BRAND_KEYWORDS.keys())

# ── LEAKAGE DETECTION KEYWORDS ────────────────────────────────────────────
# Detects posts describing diaper leakage incidents

LEAKAGE_KEYWORDS = '|'.join([
    'leaking', 'leaked', 'leaks',
    'soaked', 'soaking', 'wet',
    'dripping', 'dripped',
    'overflow', 'overflowed',
    'backflow', 'drenched',
    'soggy', 'flooded', 'seeping'
])

# ── RISK / CONTAMINATION KEYWORDS ─────────────────────────────────────────
# Detects posts mentioning safety hazards or product contamination

RISK_KEYWORDS = [
    'foreign object', 'contamination', 'rust', 'rust water',
    'adhesive', 'mold', 'mould', 'insect', 'bug', 'fly',
    'metal particle', 'metal fragment',
    'hazardous substance', 'carcinogen', 'toxic',
    'VOC', 'benzene', 'toluene', 'styrene', 'xylene', 'TVOC'
]


In [ ]:
# ── SCHEDULE CHECK AND DATE RANGE ────────────────────────────────────────
# Set environment variables: SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT,
#                             SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE, OPENAI_API_KEY

now = datetime.now() + timedelta(hours=9)  # KST

# Pipeline is scheduled to run on Tuesdays only
if now.weekday() != 1:
    print('Not Tuesday. Skipping run.')
    exit()

# Date range: previous Sunday to Saturday
last_sunday   = now - timedelta(days=now.weekday() + 8)
last_saturday = now - timedelta(days=now.weekday() + 2)
start_date = last_sunday.strftime('%Y-%m-%d')
end_date   = last_saturday.strftime('%Y-%m-%d')
print(f'Processing period: {start_date} to {end_date}')

In [ ]:
con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

In [ ]:
# ── FETCH POSTS FOR TARGET WEEK ──────────────────────────────────────────

cur.execute(f"""
SELECT * FROM NLP_TABLE
WHERE WDATE BETWEEN '{start_date}' AND '{end_date}'
AND NOT (BOARD_PATH IN ('Deal Board') OR BOARD_PATH LIKE '%HotDeal%');
""")
content_columns = [desc[0] for desc in cur.description]
content = pd.DataFrame(cur.fetchall(), columns=content_columns)
content['CONTENTS'] = content['CONTENTS'].fillna('')
content['WDATE'] = pd.to_datetime(content['WDATE'])

# Week start (Sunday-based) for grouping
content['PERIOD_START'] = content['WDATE'] - pd.to_timedelta(
    (content['WDATE'].dt.weekday + 1) % 7, unit='d'
)

In [ ]:
# ── GPT-4 WEEKLY SUMMARY ─────────────────────────────────────────────────

openai.api_key = os.environ['OPENAI_API_KEY']
GPT_MODEL = 'gpt-4o-2024-08-06'

def gpt_summary(combined_contents, past_contents, brand):
    """Generate concise summary of new brand-related insights for the week."""
    prompt = f"""
    As a marketer for {brand}, identify new insights from this week's posts
    not covered in the past 4 weeks. Summary must be concise (under 200 characters).

    Past 4 weeks:
    {past_contents}

    This week:
    {combined_contents}

    Focus on {brand}-specific insights only. Exclude other brands.
    """
    messages = [
        {'role': 'system', 'content': f'You are a marketer for {brand} analyzing customer feedback.'},
        {'role': 'user', 'content': prompt}
    ]
    return openai.ChatCompletion.create(
        model=GPT_MODEL, messages=messages
    )['choices'][0]['message']['content']

In [ ]:
# ── GENERATE WEEKLY SUMMARIES PER BRAND ──────────────────────────────────

summary_results = []

for brand in TARGET_BRANDS:
    print(f'Processing: {brand}')
    brand_content = content[content['BRAND'] == brand]
    week_start = brand_content['PERIOD_START'].min()

    while week_start <= brand_content['WDATE'].max():
        week_end = week_start + timedelta(days=6)
        week_data = brand_content[
            (brand_content['WDATE'] >= week_start) &
            (brand_content['WDATE'] <= week_end)
        ]
        if not week_data.empty:
            # Use top half of posts by view count
            top_posts = week_data.sort_values('VIEWS', ascending=False).head(len(week_data) // 2)
            combined_contents = ' '.join(top_posts['CONTENTS'].tolist())
            summary_text = gpt_summary(combined_contents, '', brand)
            summary_results.append({
                'BRAND': brand,
                'START_DATE': week_start.strftime('%Y-%m-%d'),
                'END_DATE': week_end.strftime('%Y-%m-%d'),
                'SUMMARY': summary_text
            })
        week_start += timedelta(days=7)

summary_df = pd.DataFrame(summary_results)
print(summary_df.head())

In [ ]:
# ── UPLOAD TO DATA WAREHOUSE ─────────────────────────────────────────────

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

with engine.connect() as con:
    summary_df.to_sql(name='SUMMARY_TABLE', con=con, if_exists='append',
                      method=pd_writer, index=False)

print(f'Uploaded {len(summary_df)} summaries')